# Experiment Configuration System

Configuration architecture for the Neural-Forecast BTC forecasting system.

**Purpose**: Define, validate, and manage experiment configurations across all horizons (h4, h8, h16, h32)

**Author**: Configuration Architect (config-architect)

**Last Updated**: 2025-01-15

In [ ]:
# MANDATORY CONSTRAINTS FOR FOUNDATION BUILDING
SAMPLE_SIZE = 100  # MAX 1000 for testing
MAX_STEPS = 100    # Full training uses 20000
N_WINDOWS = 2      # Full CV uses 6-10
BATCH_SIZE = 32    # Full training uses 512

import yaml
import json
from pathlib import Path
from typing import Dict, List, Any, Optional
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from dataclasses import dataclass, field
from pprint import pprint

# Project paths
PROJECT_ROOT = Path('/Users/mac-main/Neural-Forecast')
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
DEFAULTS_PATH = EXPERIMENTS_DIR / 'defaults.yaml'

print(f"Project root: {PROJECT_ROOT}")
print(f"Experiments directory: {EXPERIMENTS_DIR}")
print(f"Foundation mode: SAMPLE_SIZE={SAMPLE_SIZE}, MAX_STEPS={MAX_STEPS}")

## 1. YAML Schema Definition

Document all required fields based on Section 4.2.B (lines 1491-1530) of `docs/forecasting_sf_plan.md`

In [ ]:
@dataclass
class ConfigSchema:
    """YAML configuration schema for NeuralForecast experiments."""
    
    # Global settings
    seed: int = 1337
    freq: str = "15min"
    h: int = field(default=16)  # horizon in timesteps
    
    # Scaler configuration
    scaler_type: Dict[str, str] = field(default_factory=lambda: {
        "default": "robust",
        "PATCHTST": "revin"
    })
    
    # Cross-validation settings
    n_windows: int = 6
    step_size: int = field(default=16)  # Should equal h
    val_size: int = 64  # 16 bars * 4 hours
    refit: bool = True
    
    # Exogenous variable lists (populated at runtime)
    hist_exog_list: List[str] = field(default_factory=list)
    futr_exog_list: List[str] = field(default_factory=list)
    stat_exog_list: List[str] = field(default_factory=list)
    
    # Model portfolio
    models: List[Dict[str, Any]] = field(default_factory=list)
    
    # Training defaults (from defaults.yaml)
    training_defaults: Dict[str, Any] = field(default_factory=lambda: {
        "learning_rate": 0.001,
        "batch_size": 512,
        "early_stop_patience_steps": 400,
        "max_steps": 20000,
        "val_check_steps": 100
    })

# Define valid model types and their specific parameters
MODEL_PARAMS = {
    "NHITS": {
        "required": ["alias", "input_size", "loss"],
        "defaults": {
            "input_size": 1024,
            "n_blocks": [1, 1, 1],
            "n_pool_kernel_size": [2, 2, 1],
            "dropout_prob_theta": 0.1
        }
    },
    "NBEATSX": {
        "required": ["alias", "input_size", "loss"],
        "defaults": {
            "input_size": 1024,
            "stack_types": ['identity', 'trend', 'seasonality'],
            "n_blocks": [1, 1, 1],
            "mlp_units": [[512, 512], [512, 512], [512, 512]],
            "dropout_prob_theta": 0.1
        }
    },
    "TIDE": {
        "required": ["alias", "input_size", "loss"],
        "defaults": {
            "input_size": 1024,
            "hidden_size": 512,
            "num_encoder_layers": 2,
            "num_decoder_layers": 2,
            "dropout": 0.1
        }
    },
    "PATCHTST": {
        "required": ["alias", "input_size", "loss"],
        "defaults": {
            "input_size": 2048,  # Double for PatchTST
            "learning_rate": 0.0005,  # Lower for PatchTST
            "patch_len": 16,
            "stride": 16,
            "n_heads": 8,
            "hidden_size": 512,
            "revin": True
        }
    }
}

# Valid horizon values
VALID_HORIZONS = [4, 8, 16, 32]

# Valid loss types
VALID_LOSSES = [
    {"kind": "studentt"},
    {"kind": "mqloss", "level": [80, 90, 95]},
    {"kind": "iqloss"}
]

print("Schema defined:")
print(f"- Valid horizons: {VALID_HORIZONS}")
print(f"- Valid models: {list(MODEL_PARAMS.keys())}")
print(f"- Valid losses: StudentT, MQLoss, IQLoss")

## 2. Config Validation Functions

Validate required keys, check parameter ranges, and ensure model compatibility

In [ ]:
def validate_config(config: Dict[str, Any], strict: bool = True) -> List[str]:
    """Validate a configuration dictionary against the schema.
    
    Args:
        config: Configuration dictionary to validate
        strict: If True, treat warnings as errors
        
    Returns:
        List of validation errors (empty if valid)
    """
    errors = []
    warnings = []
    
    # Check required top-level keys
    required_keys = ['seed', 'freq', 'h', 'scaler_type', 'n_windows', 
                     'step_size', 'val_size', 'models']
    for key in required_keys:
        if key not in config:
            errors.append(f"Missing required key: {key}")
    
    # Validate horizon
    if 'h' in config:
        if config['h'] not in VALID_HORIZONS:
            errors.append(f"Invalid horizon {config['h']}. Must be one of {VALID_HORIZONS}")
        
        # Check step_size matches horizon
        if 'step_size' in config and config['step_size'] != config['h']:
            warnings.append(f"step_size ({config['step_size']}) should equal h ({config['h']})")
    
    # Validate frequency
    if 'freq' in config and config['freq'] != "15min":
        errors.append(f"freq must be '15min', got '{config['freq']}'")
    
    # Validate seed
    if 'seed' in config and config['seed'] != 1337:
        warnings.append(f"seed should be 1337 for reproducibility, got {config['seed']}")
    
    # Validate scaler_type
    if 'scaler_type' in config:
        if not isinstance(config['scaler_type'], dict):
            errors.append("scaler_type must be a dictionary")
        elif 'default' not in config['scaler_type']:
            errors.append("scaler_type must have 'default' key")
        elif config['scaler_type'].get('PATCHTST') != 'revin':
            warnings.append("PatchTST should use 'revin' scaler")
    
    # Validate models
    if 'models' in config:
        if not isinstance(config['models'], list):
            errors.append("'models' must be a list")
        else:
            for i, model_dict in enumerate(config['models']):
                errors.extend(validate_model(model_dict, i))
    
    # Validate cross-validation settings
    if 'n_windows' in config:
        if not (2 <= config['n_windows'] <= 10):
            warnings.append(f"n_windows={config['n_windows']} outside typical range [2, 10]")
    
    if 'val_size' in config:
        if config['val_size'] != 64:
            warnings.append(f"val_size={config['val_size']} differs from standard 64 (4 hours)")
    
    # In strict mode, treat warnings as errors
    if strict:
        errors.extend(warnings)
    else:
        for warning in warnings:
            print(f"⚠️  Warning: {warning}")
    
    return errors


def validate_model(model_dict: Dict[str, Any], index: int) -> List[str]:
    """Validate a single model configuration.
    
    Args:
        model_dict: Model configuration dictionary
        index: Model index in the list
        
    Returns:
        List of validation errors
    """
    errors = []
    
    if not model_dict:
        errors.append(f"Model {index}: empty configuration")
        return errors
    
    # Get model type (first key)
    model_type = list(model_dict.keys())[0]
    
    if model_type not in MODEL_PARAMS:
        errors.append(f"Model {index}: unknown type '{model_type}'")
        return errors
    
    model_config = model_dict[model_type]
    required_params = MODEL_PARAMS[model_type]["required"]
    
    # Check required parameters
    for param in required_params:
        if param not in model_config:
            errors.append(f"Model {index} ({model_type}): missing required param '{param}'")
    
    # Validate specific parameters
    if 'input_size' in model_config:
        expected = 2048 if model_type == 'PATCHTST' else 1024
        if model_config['input_size'] != expected:
            errors.append(f"Model {index} ({model_type}): input_size should be {expected}")
    
    if 'learning_rate' in model_config:
        expected = 0.0005 if model_type == 'PATCHTST' else 0.001
        if model_config['learning_rate'] != expected:
            errors.append(f"Model {index} ({model_type}): learning_rate should be {expected}")
    
    if 'loss' in model_config:
        if not isinstance(model_config['loss'], dict) or 'kind' not in model_config['loss']:
            errors.append(f"Model {index} ({model_type}): loss must be a dict with 'kind' key")
    
    return errors


# Test validation on a simple config
test_config = {
    "seed": 1337,
    "freq": "15min",
    "h": 16,
    "scaler_type": {"default": "robust", "PATCHTST": "revin"},
    "n_windows": 6,
    "step_size": 16,
    "val_size": 64,
    "models": [
        {"NHITS": {"alias": "test", "input_size": 1024, "loss": {"kind": "studentt"}}}
    ]
}

errors = validate_config(test_config, strict=False)
if errors:
    print("❌ Validation errors:")
    for error in errors:
        print(f"  - {error}")
else:
    print("✅ Test config is valid!")

## 3. Interactive Config Builder

Widgets for horizon selection, model parameter controls, and loss configuration

In [ ]:
class InteractiveConfigBuilder:
    """Interactive configuration builder with widgets."""
    
    def __init__(self):
        self.config = {}
        self.setup_widgets()
    
    def setup_widgets(self):
        """Create interactive widgets for configuration."""
        # Horizon selector
        self.horizon_widget = widgets.Dropdown(
            options=VALID_HORIZONS,
            value=16,
            description='Horizon (h):',
            style={'description_width': 'initial'}
        )
        
        # CV settings
        self.n_windows_widget = widgets.IntSlider(
            value=6, min=2, max=10, step=1,
            description='CV Windows:',
            style={'description_width': 'initial'}
        )
        
        self.val_size_widget = widgets.IntSlider(
            value=64, min=16, max=128, step=16,
            description='Val Size:',
            style={'description_width': 'initial'}
        )
        
        # Model selection
        self.model_checkboxes = {
            model: widgets.Checkbox(value=True, description=model)
            for model in MODEL_PARAMS.keys()
        }
        
        # Loss selection
        self.loss_widget = widgets.RadioButtons(
            options=['StudentT', 'MQLoss', 'IQLoss'],
            value='StudentT',
            description='Loss Type:',
            style={'description_width': 'initial'}
        )
        
        # Training parameters (for foundation mode)
        self.batch_size_widget = widgets.Dropdown(
            options=[16, 32, 64, 128, 256, 512],
            value=32 if SAMPLE_SIZE <= 100 else 512,
            description='Batch Size:',
            style={'description_width': 'initial'}
        )
        
        self.max_steps_widget = widgets.IntText(
            value=MAX_STEPS,
            description='Max Steps:',
            style={'description_width': 'initial'}
        )
        
        # Build button
        self.build_button = widgets.Button(
            description='Build Config',
            button_style='success',
            icon='check'
        )
        self.build_button.on_click(self.build_config)
        
        # Output area
        self.output = widgets.Output()
    
    def build_config(self, b=None):
        """Build configuration from widget values."""
        with self.output:
            clear_output()
            
            # Build base config
            h = self.horizon_widget.value
            self.config = {
                "seed": 1337,
                "freq": "15min",
                "h": h,
                "scaler_type": {
                    "default": "robust",
                    "PATCHTST": "revin"
                },
                "n_windows": self.n_windows_widget.value,
                "step_size": h,  # Match horizon
                "val_size": self.val_size_widget.value,
                "refit": True,
                "hist_exog_list": [],
                "futr_exog_list": [],
                "stat_exog_list": [],
                "models": []
            }
            
            # Add selected models
            loss_type = self.loss_widget.value.lower()
            loss_config = {"kind": loss_type.replace('loss', '')}
            if loss_type == 'mqloss':
                loss_config["level"] = [80, 90, 95]
            
            for model_type, checkbox in self.model_checkboxes.items():
                if checkbox.value:
                    model_config = MODEL_PARAMS[model_type]["defaults"].copy()
                    model_config["alias"] = f"{model_type}_h{h}_{loss_type[:1].upper()}"
                    model_config["loss"] = loss_config.copy()
                    model_config["batch_size"] = self.batch_size_widget.value
                    model_config["max_steps"] = self.max_steps_widget.value
                    model_config["val_check_steps"] = min(100, self.max_steps_widget.value // 10)
                    model_config["early_stop_patience_steps"] = min(400, self.max_steps_widget.value // 5)
                    
                    # Override learning rate for PatchTST
                    if model_type == 'PATCHTST' and 'learning_rate' not in model_config:
                        model_config["learning_rate"] = 0.0005
                    elif 'learning_rate' not in model_config:
                        model_config["learning_rate"] = 0.001
                    
                    self.config["models"].append({model_type: model_config})
            
            # Validate the config
            errors = validate_config(self.config, strict=False)
            
            if errors:
                print("❌ Validation errors:")
                for error in errors:
                    print(f"  - {error}")
            else:
                print("✅ Configuration is valid!\n")
                print("📋 Generated Config:")
                print(yaml.dump(self.config, default_flow_style=False, sort_keys=False))
    
    def display(self):
        """Display the interactive builder."""
        # Layout widgets
        config_section = widgets.VBox([
            widgets.HTML("<h3>Global Settings</h3>"),
            self.horizon_widget,
            widgets.HTML("<h3>Cross-Validation</h3>"),
            self.n_windows_widget,
            self.val_size_widget,
            widgets.HTML("<h3>Training Parameters</h3>"),
            self.batch_size_widget,
            self.max_steps_widget,
        ])
        
        model_section = widgets.VBox([
            widgets.HTML("<h3>Model Selection</h3>"),
            *self.model_checkboxes.values(),
            widgets.HTML("<h3>Loss Function</h3>"),
            self.loss_widget,
        ])
        
        main_layout = widgets.HBox([config_section, model_section])
        
        display(widgets.VBox([
            widgets.HTML("<h2>Interactive Config Builder</h2>"),
            main_layout,
            self.build_button,
            self.output
        ]))

# Create and display the builder
builder = InteractiveConfigBuilder()
builder.display()

## 4. Load and Test Existing Configs

Load h4.yaml, h8.yaml, h16.yaml, h32.yaml and validate their structure

In [ ]:
def load_config(horizon: int) -> Dict[str, Any]:
    """Load configuration for a specific horizon.
    
    Args:
        horizon: Horizon value (4, 8, 16, 32)
        
    Returns:
        Configuration dictionary
    """
    config_path = EXPERIMENTS_DIR / f"h{horizon}.yaml"
    
    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found: {config_path}")
    
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    return config


def test_all_configs():
    """Test all horizon configurations."""
    results = {}
    
    for horizon in VALID_HORIZONS:
        print(f"\n{'='*50}")
        print(f"Testing h{horizon}.yaml")
        print('='*50)
        
        try:
            config = load_config(horizon)
            errors = validate_config(config, strict=False)
            
            results[f"h{horizon}"] = {
                "status": "✅ Valid" if not errors else "❌ Invalid",
                "errors": errors,
                "n_models": len(config.get('models', [])),
                "step_size": config.get('step_size'),
                "n_windows": config.get('n_windows')
            }
            
            print(f"Status: {results[f'h{horizon}']['status']}")
            print(f"Models: {results[f'h{horizon}']['n_models']}")
            print(f"Step size: {results[f'h{horizon}']['step_size']}")
            print(f"CV windows: {results[f'h{horizon}']['n_windows']}")
            
            if errors:
                print("\nErrors found:")
                for error in errors:
                    print(f"  - {error}")
        
        except FileNotFoundError as e:
            results[f"h{horizon}"] = {
                "status": "⚠️  Missing",
                "errors": [str(e)]
            }
            print(f"Status: {results[f'h{horizon}']['status']}")
        
        except Exception as e:
            results[f"h{horizon}"] = {
                "status": "❌ Error",
                "errors": [str(e)]
            }
            print(f"Status: {results[f'h{horizon}']['status']}")
            print(f"Error: {e}")
    
    # Summary
    print(f"\n{'='*50}")
    print("SUMMARY")
    print('='*50)
    
    df_results = pd.DataFrame(results).T
    display(df_results)
    
    return results

# Test all configurations
test_results = test_all_configs()

## 5. Generate Missing Configs

Create h16.yaml and h32.yaml if missing, ensuring consistency across horizons

In [ ]:
def generate_horizon_config(horizon: int, force: bool = False) -> Dict[str, Any]:
    """Generate configuration for a specific horizon.
    
    Args:
        horizon: Horizon value (4, 8, 16, 32)
        force: If True, overwrite existing config
        
    Returns:
        Generated configuration dictionary
    """
    config_path = EXPERIMENTS_DIR / f"h{horizon}.yaml"
    
    if config_path.exists() and not force:
        print(f"Config h{horizon}.yaml already exists. Use force=True to overwrite.")
        return load_config(horizon)
    
    # Load defaults
    with open(DEFAULTS_PATH, 'r') as f:
        defaults = yaml.safe_load(f)
    
    # Build config based on defaults
    config = {
        "seed": defaults["seed"],
        "freq": defaults["freq"],
        "h": horizon,
        "scaler_type": defaults["scaler_type"],
        "n_windows": defaults["n_windows"],
        "step_size": horizon,  # Match horizon
        "val_size": defaults["val_size"],
        "refit": defaults["refit"],
        "hist_exog_list": [],
        "futr_exog_list": [],
        "stat_exog_list": [],
        "models": []
    }
    
    # Add models with StudentT loss
    for model_type in ['NHITS', 'NBEATSX', 'TIDE', 'PATCHTST']:
        model_config = defaults["model_defaults"][model_type].copy()
        model_config.update(defaults["training_defaults"])
        
        # Model-specific overrides
        if model_type == 'PATCHTST':
            model_config["learning_rate"] = 0.0005
            model_config["alias"] = f"PatchTST_t2048_T"
        else:
            model_config["alias"] = f"{model_type}_t1024_T"
        
        model_config["loss"] = {"kind": "studentt"}
        
        config["models"].append({model_type: model_config})
    
    # Validate before saving
    errors = validate_config(config, strict=True)
    if errors:
        print(f"❌ Generated config has errors:")
        for error in errors:
            print(f"  - {error}")
        return config
    
    # Save config
    header = f"""# Experiment Configuration for h={horizon} ({horizon * 15 // 60} hour{'s' if horizon > 4 else ''} horizon)
# BTC Intraday Forecasting with NeuralForecast Model Factory

"""
    
    with open(config_path, 'w') as f:
        f.write(header)
        # Write config sections in order
        f.write("# Global settings\n")
        for key in ['seed', 'freq', 'h']:
            if key == 'h':
                f.write(f"{key}: {config[key]}  # {horizon * 15 // 60} hour{'s' if horizon > 4 else ''} horizon ({horizon} * 15min)\n")
            else:
                f.write(f"{key}: {yaml.dump(config[key], default_flow_style=True).strip()}\n")
        
        f.write("\n# Scaler configuration with per-model overrides\n")
        f.write("scaler_type:\n")
        for k, v in config['scaler_type'].items():
            f.write(f"  {k}: {v}\n")
        
        f.write("\n# Cross-validation settings\n")
        for key in ['n_windows', 'step_size', 'val_size', 'refit']:
            f.write(f"{key}: {config[key]}\n")
        
        f.write("\n# Exogenous variable lists (populated at runtime by feature engineering)\n")
        for key in ['hist_exog_list', 'futr_exog_list', 'stat_exog_list']:
            f.write(f"{key}: []\n")
        
        f.write("\n# Model portfolio configuration\n")
        f.write("models:\n")
        for model_dict in config['models']:
            model_type = list(model_dict.keys())[0]
            f.write(f"  - {model_type}:\n")
            for k, v in model_dict[model_type].items():
                if k == 'learning_rate' and model_type == 'PATCHTST':
                    f.write(f"      {k}: {v}  # Slightly lower for PatchTST\n")
                else:
                    f.write(f"      {k}: {yaml.dump(v, default_flow_style=True).strip()}\n")
            f.write("      \n")
    
    print(f"✅ Generated h{horizon}.yaml successfully!")
    return config


# Check which configs need generation
print("Checking for missing configurations...\n")
for horizon in VALID_HORIZONS:
    config_path = EXPERIMENTS_DIR / f"h{horizon}.yaml"
    if config_path.exists():
        print(f"✅ h{horizon}.yaml exists")
    else:
        print(f"⚠️  h{horizon}.yaml missing - generating...")
        generate_horizon_config(horizon)

## 6. Export Configurations

Save validated configs and generate documentation

In [ ]:
def export_config_documentation():
    """Generate documentation for all configurations."""
    doc_lines = [
        "# Experiment Configuration Documentation",
        "",
        "## Overview",
        "This document describes the configuration system for the Neural-Forecast BTC forecasting project.",
        "",
        "## Configuration Files",
        ""
    ]
    
    # Document each horizon config
    for horizon in VALID_HORIZONS:
        try:
            config = load_config(horizon)
            doc_lines.extend([
                f"### h{horizon}.yaml",
                f"- **Horizon**: {horizon} steps ({horizon * 15 // 60} hour{'s' if horizon > 4 else ''})",
                f"- **Models**: {len(config.get('models', []))} models",
                f"- **CV Windows**: {config.get('n_windows', 'N/A')}",
                f"- **Step Size**: {config.get('step_size', 'N/A')}",
                f"- **Validation Size**: {config.get('val_size', 'N/A')} bars",
                ""
            ])
            
            # List models
            doc_lines.append("**Models:**")
            for model_dict in config.get('models', []):
                model_type = list(model_dict.keys())[0]
                model_config = model_dict[model_type]
                doc_lines.append(f"- {model_type}: {model_config.get('alias', 'unnamed')}")
            doc_lines.append("")
            
        except Exception as e:
            doc_lines.extend([
                f"### h{horizon}.yaml",
                f"- **Status**: Error loading config",
                f"- **Error**: {e}",
                ""
            ])
    
    # Add parameter reference
    doc_lines.extend([
        "## Parameter Reference",
        "",
        "### Global Parameters",
        "- `seed`: Random seed for reproducibility (1337)",
        "- `freq`: Data frequency (15min)",
        "- `h`: Forecast horizon in timesteps",
        "",
        "### Cross-Validation Parameters",
        "- `n_windows`: Number of CV windows (6-10)",
        "- `step_size`: Step between CV windows (should equal h)",
        "- `val_size`: Validation set size in bars (64 = 16 hours)",
        "- `refit`: Whether to refit on all data after CV (true)",
        "",
        "### Model Parameters",
        "- `input_size`: Historical window size (1024 for most, 2048 for PatchTST)",
        "- `learning_rate`: Learning rate (0.001 default, 0.0005 for PatchTST)",
        "- `batch_size`: Training batch size (512 for production, 32 for testing)",
        "- `max_steps`: Maximum training steps (20000 for production, 100 for testing)",
        "- `early_stop_patience_steps`: Early stopping patience (400)",
        "",
        "### Loss Functions",
        "- `StudentT`: Heavy-tailed distribution for robust predictions",
        "- `MQLoss`: Multi-quantile loss for coverage levels [80, 90, 95]",
        "- `IQLoss`: Implicit quantile loss",
        ""
    ])
    
    # Save documentation
    doc_path = EXPERIMENTS_DIR / "CONFIG_DOCUMENTATION.md"
    with open(doc_path, 'w') as f:
        f.write('\n'.join(doc_lines))
    
    print(f"✅ Documentation exported to {doc_path}")
    return doc_lines


def export_config_summary():
    """Export a summary of all configurations as CSV."""
    data = []
    
    for horizon in VALID_HORIZONS:
        try:
            config = load_config(horizon)
            
            # Extract model info
            models = config.get('models', [])
            model_names = [list(m.keys())[0] for m in models]
            
            data.append({
                'horizon': horizon,
                'hours': horizon * 15 / 60,
                'n_models': len(models),
                'models': ', '.join(model_names),
                'n_windows': config.get('n_windows'),
                'step_size': config.get('step_size'),
                'val_size': config.get('val_size'),
                'seed': config.get('seed')
            })
        except Exception as e:
            data.append({
                'horizon': horizon,
                'hours': horizon * 15 / 60,
                'error': str(e)
            })
    
    df = pd.DataFrame(data)
    csv_path = EXPERIMENTS_DIR / "config_summary.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"✅ Summary exported to {csv_path}")
    display(df)
    return df


# Export documentation and summary
print("Exporting configuration documentation...\n")
doc = export_config_documentation()
print("\nExporting configuration summary...\n")
summary_df = export_config_summary()

## Summary

This notebook has:
1. ✅ Defined the YAML schema for experiment configurations
2. ✅ Created validation functions to ensure config correctness
3. ✅ Built an interactive widget-based config builder
4. ✅ Tested all existing horizon configurations (h4, h8, h16, h32)
5. ✅ Generated any missing configurations
6. ✅ Exported documentation and summaries

All configurations are now ready for use by the model factory and training pipeline.